# ENOE 2026 T1 — Verificación de llaves y construcción de un dataset unificado

**Contexto.** Insumo para un modelo de estimación de ingresos en un buró de crédito.
Este notebook (a) verifica empíricamente si los 5 datasets principales de la ENOE se pueden
unir, (b) construye un dataset unificado a nivel persona con variables de vivienda, hogar,
cuestionario y sociodemográficas, y (c) genera agregados geográficos (entidad / municipio /
tamaño de localidad) listos para enriquecer expedientes de crédito.

---

## 1. ¿Es posible la unión? Sí — el modelo entidad-relación de la ENOE es jerárquico

La ENOE es una encuesta de hogares con diseño **vivienda → hogar → persona**. Las cinco tablas
comparten un identificador compuesto que se va extendiendo conforme se baja de nivel:

```
VIV  (vivienda)      llave = cd_a + cve_ent + con + v_sel + tipo + mes_cal
  └── HOG  (hogar)   llave = llave_viv + n_hog + h_mud
        └── SDEM     llave = llave_hog + n_ren          (todos los residentes)
              ├── COE1  llave = llave_sdem              (solo 12 años y más)
              └── COE2  llave = llave_sdem              (solo 12 años y más)
```

> **Nombres verificados contra los diccionarios de la ENOE 2026 1T.** Ojo: la entidad se llama
> `cve_ent`, no `ent`; el municipio es `cve_mun`, no `mun`; el tamaño de localidad viene
> desdoblado en `t_loc_tri` / `t_loc_men` y el estrato de diseño en `est_d_tri` / `est_d_men`.
> Las cinco tablas traen además `cvegeo` (5 dígitos = entidad + municipio), que es la llave
> geoestadística estándar del INEGI y el mejor puente hacia CONEVAL, CONAPO o cualquier
> catálogo municipal.

Cardinalidades esperadas:

| Unión | Tipo | Nota |
|---|---|---|
| SDEM → COE1 | 1:1 | `left`: COE1 solo cubre 12+ con entrevista completa |
| SDEM → COE2 | 1:1 | igual que COE1 |
| SDEM → HOG | m:1 | varias personas por hogar |
| HOG → VIV | m:1 | puede haber más de un hogar por vivienda (raro, ~1%) |

**Advertencia importante:** los componentes de la llave son *códigos con ceros a la izquierda*
(`con` = 5 dígitos, `n_ren` = 2 dígitos, etc.). Si pandas los lee como enteros en un archivo y
como texto en otro, el `merge` devuelve 0 filas o —peor— coincidencias parciales silenciosas.
El notebook normaliza todas las llaves a una representación canónica antes de unir.

**Segunda advertencia:** VIV, HOG, COE1 y COE2 **reutilizan los mismos nombres de variable**
(`p1`, `p2`, `p3`, …) con significados completamente distintos. Sin prefijar, el merge produce
`p1_x`, `p1_y`, `p1_z` y se pierde la trazabilidad. Aquí se prefija cada módulo.

**Tercera advertencia, y la más relevante para un modelo de ingresos:** el ingreso declarado
**no está en COE1**. COE1 cubre las preguntas 1 a 5h (condición de actividad, búsqueda, tipo de
unidad económica, jornada). Las preguntas de ingreso viven en **COE2**, y hay una trampa de
nomenclatura: `p6b1` es la *periodicidad* del pago (escala 1-8), mientras que el **monto en pesos
es `p6b2`**. Confundirlas produce un modelo entrenado sobre un código ordinal de frecuencia.

> El notebook no asume que estos supuestos sean ciertos: los **prueba** contra tus archivos y
> reporta cualquier desviación (sección 4 y 5). Si INEGI cambió algo en 2026 T1, lo vas a ver.

## 2. Configuración

Ajusta `BASE` a la ruta local donde descomprimiste `conjunto_de_datos_enoe_2026_1t_csv/`.

In [7]:
import os

In [11]:
os.getcwd()

'/Users/r/Projects/datos_EI'

In [13]:
from pathlib import Path
import re, json, warnings
import numpy as np
import pandas as pd

# ---------------------------------------------------------------- AJUSTA ESTO
BASE = Path("conjunto_de_datos_enoe_2026_1t_csv")   # carpeta raíz descomprimida
ANIO, TRIM = 2026, 1
OUT  = Path("./salida_enoe"); OUT.mkdir(exist_ok=True, parents=True)
# -----------------------------------------------------------------------------

pd.set_option("display.max_columns", 250)
pd.set_option("display.width", 200)
warnings.filterwarnings("ignore", category=FutureWarning)

MODULOS = ["viv", "hog", "sdem", "coe1", "coe2"]

# Llaves esperadas — nombres verificados contra los diccionarios de la ENOE 2026 1T.
# OJO: es cve_ent (no ent) y cve_mun (no mun).
K_VIV = ["cd_a", "cve_ent", "con", "v_sel", "tipo", "mes_cal"]
K_HOG = K_VIV + ["n_hog", "h_mud"]
K_PER = K_HOG + ["n_ren"]

# Candidatas a desempate si la llave base no es única en tus archivos
K_EXTRA = ["upm", "d_sem", "n_pro_viv", "n_ent", "per", "ur"]

# Variables geográficas y de diseño, por si cambian de nombre entre trimestres
GEO = {"entidad": "cve_ent", "municipio": "cve_mun", "cvegeo": "cvegeo",
       "localidad": "cve_loc", "ageb": "cve_ageb"}
CAND_TLOC = ["t_loc_tri", "t_loc_men", "t_loc"]      # tamaño de localidad
CAND_ESTD = ["est_d_tri", "est_d_men", "est_d"]      # estrato de diseño
CAND_FAC  = ["fac_tri", "fac_men", "fac"]            # factor de expansión

LLAVES = {"viv": K_VIV, "hog": K_HOG, "sdem": K_PER, "coe1": K_PER, "coe2": K_PER}
PREFIJO = {"viv": "viv_", "hog": "hog_", "coe1": "c1_", "coe2": "c2_"}  # sdem = espina, sin prefijo

assert BASE.exists(), f"No existe {BASE.resolve()} — ajusta la variable BASE."
print("BASE =", BASE.resolve())

BASE = /Users/r/Projects/datos_EI/conjunto_de_datos_enoe_2026_1t_csv


## 3. Descubrimiento y carga

Se recorre la estructura `conjunto_de_datos_<mod>_enoe_2026_1t/{conjunto_de_datos,diccionario_de_datos,catalogos}/`.
La lectura fuerza `str` en las columnas de llave para preservar ceros a la izquierda, y prueba
varias codificaciones (INEGI suele publicar en `latin-1`).

In [14]:
TODAS_LLAVES = sorted(set(K_PER + K_EXTRA))

def _subdir(d: Path, nombre: str):
    for p in d.iterdir():
        if p.is_dir() and p.name.lower() == nombre:
            return p
    return None

def localizar(base: Path) -> dict:
    reg = {}
    for m in MODULOS:
        pat = re.compile(rf"(?:^|_){m}(?:_|$)")
        cands = [p for p in base.iterdir() if p.is_dir() and pat.search(p.name.lower())]
        if not cands:
            raise FileNotFoundError(f"No encontré carpeta del módulo '{m}' dentro de {base}")
        d = sorted(cands, key=lambda p: len(p.name))[0]
        cd = _subdir(d, "conjunto_de_datos")
        dd = _subdir(d, "diccionario_de_datos")
        ct = _subdir(d, "catalogos")
        csvs = sorted(cd.glob("*.csv")) if cd else []
        if not csvs:
            raise FileNotFoundError(f"Sin .csv en {cd}")
        reg[m] = {
            "dir": d,
            "datos": csvs[0],
            "dicc": sorted(dd.glob("*.csv"))[0] if dd and list(dd.glob("*.csv")) else None,
            "catalogos": sorted(ct.glob("*.csv")) if ct else [],
        }
    return reg

def leer_csv(path: Path, **kw) -> pd.DataFrame:
    ultimo = None
    for enc in ("utf-8", "latin-1", "cp1252"):
        try:
            return pd.read_csv(path, encoding=enc, low_memory=False, **kw)
        except UnicodeDecodeError as e:
            ultimo = e
    raise ultimo

def cargar(path: Path) -> pd.DataFrame:
    # 1) leer solo el encabezado para saber qué llaves existen realmente
    head = leer_csv(path, nrows=0)
    cols = {c.strip().lower(): c for c in head.columns}
    dtypes = {cols[k]: "string" for k in TODAS_LLAVES if k in cols}
    df = leer_csv(path, dtype=dtypes)
    df.columns = [c.strip().lower() for c in df.columns]
    return df

REG = localizar(BASE)
inv = pd.DataFrame([
    {"modulo": m,
     "archivo_datos": r["datos"].name,
     "MB": round(r["datos"].stat().st_size / 1e6, 1),
     "diccionario": r["dicc"].name if r["dicc"] else "—",
     "n_catalogos": len(r["catalogos"])}
    for m, r in REG.items()
])
display(inv)

,modulo,archivo_datos,MB,diccionario,n_catalogos
0,viv,conjunto_de_datos_viv_enoe_2026_1t.csv,13.4,diccionario_datos_viv_enoe_2026_1t.csv,13
1,hog,conjunto_de_datos_hog_enoe_2026_1t.csv,17.6,diccionario_datos_hog_enoe_2026_1t.csv,22
2,sdem,conjunto_de_datos_sdem_enoe_2026_1t.csv,119.1,diccionario_datos_sdem_enoe_2026_1t.csv,89
3,coe1,conjunto_de_datos_coe1_enoe_2026_1t.csv,143.8,diccionario_datos_coe1_enoe_2026_1t.csv,135
4,coe2,conjunto_de_datos_coe2_enoe_2026_1t.csv,107.4,diccionario_datos_coe2_enoe_2026_1t.csv,104


In [15]:
DF = {}
for m in MODULOS:
    DF[m] = cargar(REG[m]["datos"])
    print(f"{m:5s}  filas={len(DF[m]):>8,}  columnas={DF[m].shape[1]:>4}")

print("\nPrimeras columnas de cada módulo:")
for m in MODULOS:
    print(f"  {m:5s}: {list(DF[m].columns[:14])}")

viv    filas= 150,362  columnas=  26
hog    filas= 151,324  columnas=  38
sdem   filas= 417,437  columnas= 115
coe1   filas= 343,771  columnas= 191
coe2   filas= 343,771  columnas= 139

Primeras columnas de cada módulo:
  viv  : ['cve_loc', 'cve_mun', 'est', 'est_d_tri', 'est_d_men', 'cve_ageb', 't_loc_tri', 't_loc_men', 'cd_a', 'cve_ent', 'con', 'upm', 'd_sem', 'n_pro_viv']
  hog  : ['cve_loc', 'cve_mun', 'est', 'est_d_tri', 'est_d_men', 'cve_ageb', 't_loc_tri', 't_loc_men', 'cd_a', 'cve_ent', 'con', 'upm', 'd_sem', 'n_pro_viv']
  sdem : ['r_def', 'cve_loc', 'cve_mun', 'est', 'est_d_tri', 'est_d_men', 'cve_ageb', 't_loc_tri', 't_loc_men', 'cd_a', 'cve_ent', 'con', 'upm', 'd_sem']
  coe1 : ['r_def', 'cd_a', 'cve_ent', 'con', 'upm', 'd_sem', 'n_pro_viv', 'v_sel', 'n_hog', 'h_mud', 'n_ent', 'per', 'n_ren', 'eda']
  coe2 : ['cd_a', 'cve_ent', 'con', 'upm', 'd_sem', 'n_pro_viv', 'v_sel', 'n_hog', 'h_mud', 'n_ent', 'per', 'n_ren', 'eda', 'n_inf']


### 3.1 Diccionarios de datos

INEGI publica los diccionarios como CSV con filas de encabezado variables. El parser busca la
fila que contiene el nombre mnemónico y arma un catálogo `modulo → variable → descripción`.
Si el formato cambió, se muestra el archivo crudo para que ajustes a mano.

In [18]:
import csv as _csv

def leer_ragged(path: Path) -> pd.DataFrame:
    # Los diccionarios de INEGI traen filas de preámbulo con distinto número de campos,
    # lo que rompe read_csv. Se parsea a mano y se rellena a la anchura máxima.
    for enc in ("utf-8", "latin-1", "cp1252"):
        try:
            with open(path, newline="", encoding=enc) as f:
                filas = [r for r in _csv.reader(f)]
            break
        except UnicodeDecodeError:
            continue
    ancho = max((len(r) for r in filas), default=0)
    filas = [r + [""] * (ancho - len(r)) for r in filas]
    return pd.DataFrame(filas, dtype=str).fillna("")

def parsear_diccionario(path: Path, modulo: str) -> pd.DataFrame:
    vacio = pd.DataFrame(columns=["modulo", "variable", "descripcion"])
    if path is None:
        return vacio
    raw = leer_ragged(path)
    if raw.empty:
        return vacio

    fila_h = None
    for i in range(min(15, len(raw))):
        celda = " ".join(raw.iloc[i].astype(str)).lower()
        if any(t in celda for t in ("nemónico", "nemonico", "mnemónico", "mnemonico",
                                    "nombre_campo", "nombre de la variable")):
            fila_h = i
            break
    if fila_h is None:
        print(f"  [!] {modulo}: no reconocí el encabezado, revisa {path.name}")
        display(raw.head(12))
        return vacio

    tab = raw.iloc[fila_h + 1:].copy()
    tab.columns = [str(c).strip().lower() for c in raw.iloc[fila_h]]
    tab = tab.loc[:, ~pd.Index(tab.columns).duplicated()]

    def _buscar(tokens, excluir=None):
        for c in tab.columns:
            if c != excluir and any(t in c for t in tokens):
                return c
        return None

    c_var = _buscar(("nemónico", "nemonico", "mnemónico", "mnemonico",
                     "nombre de la variable"))
    c_des = _buscar(("nombre_campo", "nombre del campo", "descripción",
                     "descripcion", "etiqueta", "campo"), excluir=c_var)
    if c_var is None:
        return vacio

    desc = tab[c_des].astype(str).str.strip() if c_des else pd.Series("", index=tab.index)
    out = pd.DataFrame({
        "modulo": modulo,
        "variable": tab[c_var].astype(str).str.strip().str.lower(),
        "descripcion": desc,
    })
    return out[out["variable"].str.match(r"^[a-z0-9_]+$", na=False)].reset_index(drop=True)

DICC = pd.concat([parsear_diccionario(REG[m]["dicc"], m) for m in MODULOS], ignore_index=True)
print(f"Variables documentadas: {len(DICC)}")
display(DICC.groupby("modulo").size().rename("n_variables").to_frame())

# Variables presentes en el CSV pero ausentes del diccionario (y viceversa)
for m in MODULOS:
    en_csv = set(DF[m].columns)
    en_dic = set(DICC.loc[DICC.modulo == m, "variable"])
    if en_dic:
        falta = sorted(en_csv - en_dic)[:12]
        if falta:
            print(f"{m:5s}: en CSV pero no en diccionario -> {falta}")

Variables documentadas: 509


,n_variables
modulo,
coe1,191
coe2,139
hog,38
sdem,115
viv,26


In [19]:
# --- catálogos de códigos (útiles para decodificar clase1, clase2, pos_ocu, scian, ...) ---
CATALOGOS = {}
for m in MODULOS:
    for p in REG[m]["catalogos"]:
        CATALOGOS[f"{m}/{p.stem}"] = p
print(f"{len(CATALOGOS)} catálogos disponibles:")
for k in sorted(CATALOGOS)[:40]:
    print("  ", k)

def ver_catalogo(nombre_parcial: str):
    hits = [k for k in CATALOGOS if nombre_parcial.lower() in k.lower()]
    for k in hits[:3]:
        print(f"\n=== {k} ===")
        display(leer_ragged(CATALOGOS[k]).head(25))
    if not hits:
        print("Sin coincidencias. Opciones:", sorted(CATALOGOS)[:20])

# Ejemplo: ver_catalogo("clase")

363 catálogos disponibles:
   coe1/cd_a
   coe1/cve_ent
   coe1/cve_mun
   coe1/eda
   coe1/h_mud
   coe1/mes_cal
   coe1/n_ent
   coe1/n_hog
   coe1/n_inf
   coe1/n_ren
   coe1/p1
   coe1/p1a1
   coe1/p1a2
   coe1/p1a3
   coe1/p1b
   coe1/p1c
   coe1/p1d
   coe1/p1e
   coe1/p2_1
   coe1/p2_2
   coe1/p2_3
   coe1/p2_4
   coe1/p2_9
   coe1/p2a_dia
   coe1/p2a_mes
   coe1/p2a_sem
   coe1/p2b
   coe1/p2b_dia
   coe1/p2b_mes
   coe1/p2b_sem
   coe1/p2c
   coe1/p2d1
   coe1/p2d10
   coe1/p2d11
   coe1/p2d2
   coe1/p2d3
   coe1/p2d4
   coe1/p2d5
   coe1/p2d6
   coe1/p2d7


## 4. Verificación de llaves (la pregunta central)

Tres pruebas, en orden:

1. **Presencia** — ¿existen todas las columnas de la llave en cada módulo?
2. **Unicidad** — ¿la llave identifica una sola fila? Si no, se busca greedy la columna extra
   que resuelve los duplicados.
3. **Integridad referencial** — ¿todo hogar apunta a una vivienda existente? ¿toda persona a un hogar?

In [20]:
# Normaliza una columna de llave a texto canónico, tolerando ceros a la izquierda
# presentes en un archivo y ausentes en otro.
def canonizar_llave(s: pd.Series) -> pd.Series:
    t = s.astype("string").str.strip()
    t = t.replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
    # si todo es numérico entero, colapsar a entero-como-texto (quita padding inconsistente)
    limpio = t.dropna()
    if len(limpio) and limpio.str.fullmatch(r"\d+").all():
        return pd.to_numeric(t, errors="coerce").astype("Int64").astype("string")
    return t

# --- prueba 1: presencia -----------------------------------------------------
pres = []
for m in MODULOS:
    faltan = [k for k in LLAVES[m] if k not in DF[m].columns]
    pres.append({"modulo": m, "llave_esperada": " + ".join(LLAVES[m]),
                 "faltantes": ", ".join(faltan) if faltan else "— ninguna —",
                 "ok": not faltan})
pres = pd.DataFrame(pres)
display(pres)
if not pres["ok"].all():
    print("[!] Revisa el modelo entidad-relación: la llave esperada no está completa.")

# --- normalizar llaves in-place ---------------------------------------------
for m in MODULOS:
    for k in set(LLAVES[m] + [c for c in K_EXTRA if c in DF[m].columns]):
        if k in DF[m].columns:
            DF[m][k] = canonizar_llave(DF[m][k])
print("Llaves normalizadas a texto canónico.")

,modulo,llave_esperada,faltantes,ok
0,viv,cd_a + cve_ent + con + v_sel + tipo + mes_cal,— ninguna —,True
1,hog,cd_a + cve_ent + con + v_sel + tipo + mes_cal ...,— ninguna —,True
2,sdem,cd_a + cve_ent + con + v_sel + tipo + mes_cal ...,— ninguna —,True
3,coe1,cd_a + cve_ent + con + v_sel + tipo + mes_cal ...,— ninguna —,True
4,coe2,cd_a + cve_ent + con + v_sel + tipo + mes_cal ...,— ninguna —,True


Llaves normalizadas a texto canónico.


In [21]:
# --- prueba 2: unicidad ------------------------------------------------------
LLAVE_FINAL = {}
res = []
for m in MODULOS:
    k = [c for c in LLAVES[m] if c in DF[m].columns]
    dups = int(DF[m].duplicated(subset=k).sum())
    añadidas = []
    while dups > 0:
        # buscar greedy la columna extra que más reduce duplicados
        mejor, mejor_dups = None, dups
        for c in K_EXTRA:
            if c in DF[m].columns and c not in k:
                d = int(DF[m].duplicated(subset=k + [c]).sum())
                if d < mejor_dups:
                    mejor, mejor_dups = c, d
        if mejor is None:
            break
        k, dups = k + [mejor], mejor_dups
        añadidas.append(mejor)
    LLAVE_FINAL[m] = k
    res.append({"modulo": m, "filas": len(DF[m]), "llave_usada": " + ".join(k),
                "extra_necesaria": ", ".join(añadidas) or "—",
                "duplicados_restantes": dups,
                "unica": dups == 0})
res = pd.DataFrame(res)
display(res)

if res["unica"].all():
    print("✔ Las 5 tablas tienen llave única. La unión es viable como 1:1 / m:1.")
else:
    print("[!] Hay duplicados irresolubles — inspecciona los casos antes de unir:")
    for m in res.loc[~res.unica, "modulo"]:
        k = LLAVE_FINAL[m]
        display(DF[m][DF[m].duplicated(subset=k, keep=False)].sort_values(k).head(10)[k])

,modulo,filas,llave_usada,extra_necesaria,duplicados_restantes,unica
0,viv,150362,cd_a + cve_ent + con + v_sel + tipo + mes_cal,—,0,True
1,hog,151324,cd_a + cve_ent + con + v_sel + tipo + mes_cal ...,—,0,True
2,sdem,417437,cd_a + cve_ent + con + v_sel + tipo + mes_cal ...,—,0,True
3,coe1,343771,cd_a + cve_ent + con + v_sel + tipo + mes_cal ...,—,0,True
4,coe2,343771,cd_a + cve_ent + con + v_sel + tipo + mes_cal ...,—,0,True


✔ Las 5 tablas tienen llave única. La unión es viable como 1:1 / m:1.


In [22]:
# --- prueba 3: integridad referencial ---------------------------------------
def integridad(hijo: str, padre: str, llave: list) -> dict:
    k = [c for c in llave if c in DF[hijo].columns and c in DF[padre].columns]
    h = DF[hijo][k].drop_duplicates()
    p = DF[padre][k].drop_duplicates()
    huerfanos = h.merge(p, on=k, how="left", indicator=True)
    n_huerf = int((huerfanos["_merge"] == "left_only").sum())
    sin_hijo = p.merge(h, on=k, how="left", indicator=True)
    n_sin = int((sin_hijo["_merge"] == "left_only").sum())
    return {"relacion": f"{hijo} → {padre}", "llave": " + ".join(k),
            "llaves_hijo": len(h), "llaves_padre": len(p),
            "huerfanos_(hijo_sin_padre)": n_huerf,
            "padres_sin_hijo": n_sin,
            "cobertura_%": round(100 * (1 - n_huerf / max(len(h), 1)), 3)}

ri = pd.DataFrame([
    integridad("hog",  "viv",  K_VIV),
    integridad("sdem", "hog",  K_HOG),
    integridad("coe1", "sdem", K_PER),
    integridad("coe2", "sdem", K_PER),
])
display(ri)

if (ri["huerfanos_(hijo_sin_padre)"] == 0).all():
    print("✔ Integridad referencial perfecta: ninguna fila hija queda sin padre.")
else:
    print("[!] Hay huérfanos. Causa habitual: distinta convención de ceros a la izquierda,\n"
          "    o archivos de trimestres distintos mezclados. Revisa antes de continuar.")

,relacion,llave,llaves_hijo,llaves_padre,huerfanos_(hijo_sin_padre),padres_sin_hijo,cobertura_%
0,hog → viv,cd_a + cve_ent + con + v_sel + tipo + mes_cal,150362,150362,0,0,100.0
1,sdem → hog,cd_a + cve_ent + con + v_sel + tipo + mes_cal ...,125222,151324,0,26102,100.0
2,coe1 → sdem,cd_a + cve_ent + con + v_sel + tipo + mes_cal ...,343771,417437,0,73666,100.0
3,coe2 → sdem,cd_a + cve_ent + con + v_sel + tipo + mes_cal ...,343771,417437,0,73666,100.0


✔ Integridad referencial perfecta: ninguna fila hija queda sin padre.


In [23]:
# --- cobertura de los cuestionarios COE (solo 12 años y más) -----------------
sd = DF["sdem"]
k = K_PER
en_coe1 = sd[k].merge(DF["coe1"][k].assign(_c1=1), on=k, how="left")["_c1"].notna()

if "eda" in sd.columns:
    edad = pd.to_numeric(sd["eda"], errors="coerce").where(lambda s: s < 99)
    cob = pd.DataFrame({"grupo_edad": pd.cut(edad, [-1, 11, 14, 17, 24, 34, 44, 54, 64, 130],
                                             labels=["0-11","12-14","15-17","18-24","25-34",
                                                     "35-44","45-54","55-64","65+"]),
                        "en_coe1": en_coe1.values})
    tab = cob.groupby("grupo_edad", observed=True)["en_coe1"].agg(["size", "sum"])
    tab["cobertura_%"] = (100 * tab["sum"] / tab["size"]).round(1)
    display(tab.rename(columns={"size": "personas_sdem", "sum": "con_coe1"}))
print(f"\nSDEM: {len(sd):,} personas · COE1: {len(DF['coe1']):,} · "
      f"cobertura global {100*en_coe1.mean():.1f}%")
print("Esperado: ~0% para 0-11 años y ~alto para 12+ (el resto son no respuesta / ausentes).")

,personas_sdem,con_coe1,cobertura_%
grupo_edad,,,
0-11,63048,0,0.0
12-14,20513,20472,99.8
15-17,21262,21209,99.8
18-24,45828,45692,99.7
25-34,58978,58790,99.7
35-44,56458,56335,99.8
45-54,53265,53089,99.7
55-64,43071,42972,99.8
65+,45314,45212,99.8



SDEM: 417,437 personas · COE1: 343,771 · cobertura global 82.4%
Esperado: ~0% para 0-11 años y ~alto para 12+ (el resto son no respuesta / ausentes).


## 5. Construcción del dataset unificado

Espina dorsal = **SDEM** (una fila por persona residente). Se le pegan COE1/COE2 (1:1),
HOG (m:1) y VIV (m:1). Todo con `validate=` para que pandas aborte si la cardinalidad
no es la esperada — nada de duplicación silenciosa de filas.

Los módulos no-SDEM se prefijan (`viv_`, `hog_`, `c1_`, `c2_`) porque comparten nombres
de variable con significados distintos.

In [24]:
def prefijar(df: pd.DataFrame, llave: list, pref: str) -> pd.DataFrame:
    ren = {c: f"{pref}{c}" for c in df.columns if c not in llave}
    return df.rename(columns=ren)

base = DF["sdem"].copy()
n0 = len(base)

pasos = [
    ("coe1", K_PER, "1:1"),
    ("coe2", K_PER, "1:1"),
    ("hog",  K_HOG, "m:1"),
    ("viv",  K_VIV, "m:1"),
]

bitacora = []
uni = base
for m, llave, card in pasos:
    k = [c for c in llave if c in uni.columns and c in DF[m].columns]
    der = prefijar(DF[m], k, PREFIJO[m])
    antes = len(uni)
    uni = uni.merge(der, on=k, how="left", validate=card)
    col_test = [c for c in der.columns if c not in k][0]
    bitacora.append({
        "modulo": m, "cardinalidad": card, "llave": " + ".join(k),
        "cols_agregadas": der.shape[1] - len(k),
        "filas_antes": antes, "filas_despues": len(uni),
        "match_%": round(100 * uni[col_test].notna().mean(), 2),
    })

display(pd.DataFrame(bitacora))
assert len(uni) == n0, f"El merge cambió el número de filas: {n0} → {len(uni)}"
print(f"\n✔ Dataset unificado: {len(uni):,} filas × {uni.shape[1]} columnas "
      f"(sin duplicación de la espina SDEM)")

,modulo,cardinalidad,llave,cols_agregadas,filas_antes,filas_despues,match_%
0,coe1,1:1,cd_a + cve_ent + con + v_sel + tipo + mes_cal ...,182,417437,417437,82.35
1,coe2,1:1,cd_a + cve_ent + con + v_sel + tipo + mes_cal ...,130,417437,417437,82.35
2,hog,m:1,cd_a + cve_ent + con + v_sel + tipo + mes_cal ...,30,417437,417437,100.00
3,viv,m:1,cd_a + cve_ent + con + v_sel + tipo + mes_cal,20,417437,417437,100.00



✔ Dataset unificado: 417,437 filas × 477 columnas (sin duplicación de la espina SDEM)


### 5.1 Limpieza de códigos centinela y filtros analíticos

INEGI codifica la no respuesta con valores centinela (`999998`, `999999`, `99`, …). Si no se
limpian, cualquier promedio de ingreso queda destruido. Además se aplica el filtro estándar de
INEGI para tabulados: `r_def == 0` (entrevista completa) y `c_res ∈ {1, 3}` (residentes habituales).

In [25]:
# Centinelas de no respuesta, tomados de la columna RANGO_CLAVES del diccionario 2026 1T.
# ingocup: 1-999998  ·  anios_esc: 1-24,99  ·  eda(sdem): 00-99  ·  eda(coe): 12-97,98
# c2_p6b2 (monto en pesos, COE2): 000001-999998, 999999
CENTINELAS = {
    "ingocup":     [999998, 999999],
    "ing_x_hrs":   [999998, 999999],
    "anios_esc":   [99],
    "eda":         [99],
    "c2_p6b2":     [999998, 999999],   # monto declarado del trabajo principal
    "c2_p7gcan":   [999998, 999999],   # monto del segundo trabajo
    "c2_p9mcan":   [999998, 999999],   # monto del trabajo anterior
}
# NOTA: hrsocup tiene rango 1-168 (sin centinela) y 'salario' es el salario mínimo mensual
# de la zona, un valor real — no los toques.

for col, vals in CENTINELAS.items():
    if col in uni.columns:
        uni[col] = pd.to_numeric(uni[col], errors="coerce").replace(vals, np.nan)

# Factor de expansión: ENOE 2026 trae fac_tri (trimestral) y fac_men (mensual)
FAC = next((c for c in CAND_FAC if c in uni.columns), None)
assert FAC, f"No encontré el factor de expansión entre {CAND_FAC}."
uni[FAC] = pd.to_numeric(uni[FAC], errors="coerce")

# Tamaño de localidad y estrato de diseño (nombres desdoblados desde 2020)
T_LOC = next((c for c in CAND_TLOC if c in uni.columns), None)
EST_D = next((c for c in CAND_ESTD if c in uni.columns), None)
print(f"Factor de expansión: {FAC} · tamaño de localidad: {T_LOC} · estrato: {EST_D}")

# --- claves geográficas con ceros a la izquierda ------------------------------
# Jalisco es '14', no '14'->14. Sin el padding, el join contra CONEVAL/CONAPO falla
# justo en las entidades 01-09 y en todo municipio con clave < 100.
ANCHOS = {"cve_ent": 2, "cve_mun": 3, "cvegeo": 5, "cve_loc": 4, "cve_ageb": 5, "cd_a": 2}
for c, w in ANCHOS.items():
    if c in uni.columns:
        uni[c] = (pd.to_numeric(uni[c], errors="coerce").astype("Int64")
                    .astype("string").str.zfill(w).replace("<NA>", pd.NA))

# cvegeo debe ser entidad(2) + municipio(3). Se reconstruye y se contrasta.
if {"cve_ent", "cve_mun"} <= set(uni.columns):
    recon = uni["cve_ent"] + uni["cve_mun"]
    if "cvegeo" in uni.columns:
        disc = (uni["cvegeo"] != recon).sum()
        print(f"cvegeo: {disc:,} filas donde no coincide con cve_ent+cve_mun "
              f"({100*disc/len(uni):.2f}%)")
    else:
        uni["cvegeo"] = recon
        print("cvegeo reconstruido a partir de cve_ent + cve_mun.")

# Ingreso en salarios mínimos mensuales: comparable entre zona fronteriza y resto del país
if {"ingocup", "salario"} <= set(uni.columns):
    sal = pd.to_numeric(uni["salario"], errors="coerce").where(lambda s: s > 0)
    uni["ingocup_sm"] = pd.to_numeric(uni["ingocup"], errors="coerce") / sal
    print("Añadida 'ingocup_sm' (ingreso en salarios mínimos mensuales de la zona salarial).")

# Filtro estándar INEGI
m_ok = pd.Series(True, index=uni.index)
if "r_def" in uni.columns:
    m_ok &= pd.to_numeric(uni["r_def"], errors="coerce").eq(0)
if "c_res" in uni.columns:
    m_ok &= pd.to_numeric(uni["c_res"], errors="coerce").isin([1, 3])

print(f"Filtro r_def==0 & c_res in (1,3): {m_ok.sum():,} de {len(uni):,} filas "
      f"({100*m_ok.mean():.1f}%)")
ana = uni[m_ok].copy()

Factor de expansión: fac_tri · tamaño de localidad: t_loc_tri · estrato: est_d_tri
cvegeo: 0 filas donde no coincide con cve_ent+cve_mun (0.00%)
Añadida 'ingocup_sm' (ingreso en salarios mínimos mensuales de la zona salarial).
Filtro r_def==0 & c_res in (1,3): 406,740 de 417,437 filas (97.4%)


In [26]:
# --- validación contra cifras publicadas de INEGI ---------------------------
def num(df, c):
    return pd.to_numeric(df[c], errors="coerce") if c in df.columns else pd.Series(np.nan, index=df.index)

pob      = ana[FAC].sum()
edad     = num(ana, "eda")
pob15    = ana.loc[edad >= 15, FAC].sum()
clase1   = num(ana, "clase1"); clase2 = num(ana, "clase2")
pea      = ana.loc[clase1.eq(1), FAC].sum()
ocupada  = ana.loc[clase2.eq(1), FAC].sum()
desocup  = ana.loc[clase2.eq(2), FAC].sum()

chk = pd.DataFrame([
    ["Población total (expandida)",      pob,              "≈ 132 millones"],
    ["Población de 15 años y más",       pob15,            "≈ 103 millones"],
    ["PEA (clase1 == 1)",                pea,              "≈ 61-62 millones"],
    ["Población ocupada (clase2 == 1)",  ocupada,          "≈ 59-60 millones"],
    ["Tasa de desocupación (%)",         100*desocup/max(pea,1), "≈ 2.5 - 3.0"],
], columns=["indicador", "valor_calculado", "orden_de_magnitud_esperado"])
chk["valor_calculado"] = chk["valor_calculado"].map(lambda v: f"{v:,.1f}")
display(chk)
print("Si estas cifras están muy fuera de rango, el factor de expansión o el filtro están mal.")

,indicador,valor_calculado,orden_de_magnitud_esperado
0,Población total (expandida),"131,150,866.0",≈ 132 millones
1,Población de 15 años y más,"104,074,365.0",≈ 103 millones
2,PEA (clase1 == 1),"61,458,425.0",≈ 61-62 millones
3,Población ocupada (clase2 == 1),"59,893,300.0",≈ 59-60 millones
4,Tasa de desocupación (%),2.5,≈ 2.5 - 3.0


Si estas cifras están muy fuera de rango, el factor de expansión o el filtro están mal.


## 6. Agregados por nivel geográfico

Aquí es donde la ENOE se vuelve directamente útil para un modelo de ingresos sobre expedientes
de crédito: no puedes unir ENOE a una persona real (es anónima), pero **sí** puedes construir
priors de ingreso y de estructura laboral por geografía y pegarlos al expediente por CP/municipio.

Se calculan a tres niveles: entidad, entidad+municipio y tamaño de localidad (`t_loc`),
todos ponderados por el factor de expansión.

In [27]:
def w_quantile(x, w, q=0.5):
    x = pd.to_numeric(x, errors="coerce"); w = pd.to_numeric(w, errors="coerce")
    m = x.notna() & w.notna() & (w > 0)
    if m.sum() == 0:
        return np.nan
    x, w = x[m].to_numpy(float), w[m].to_numpy(float)
    o = np.argsort(x); x, w = x[o], w[o]
    cw = np.cumsum(w) - 0.5 * w
    return float(np.interp(q, cw / w.sum(), x))

def w_mean(x, w):
    x = pd.to_numeric(x, errors="coerce"); w = pd.to_numeric(w, errors="coerce")
    m = x.notna() & w.notna() & (w > 0)
    return float(np.average(x[m], weights=w[m])) if m.any() else np.nan

def w_share(mask, w, universo=None):
    w = pd.to_numeric(w, errors="coerce")
    u = w if universo is None else w.where(universo, 0)
    den = u.sum()
    return float(w.where(mask & (universo if universo is not None else True), 0).sum() / den) if den else np.nan

def resumen_grupo(d: pd.DataFrame) -> dict:
    w   = d[FAC]
    c2  = num(d, "clase2"); c1 = num(d, "clase1")
    ocu = c2.eq(1)
    ing = num(d, "ingocup").where(lambda s: s > 0)
    r = {
        "n_muestra":       len(d),
        "pob_expandida":   float(w.sum()),
        "pob_ocupada":     float(w.where(ocu, 0).sum()),
        "tasa_desocup":    w_share(c2.eq(2), w, c1.eq(1)),
        "tasa_particip":   w_share(c1.eq(1), w, num(d, "eda").ge(15)),
        "ing_medio_ocup":  w_mean(ing.where(ocu), w),
        "ing_mediana_ocup": w_quantile(ing.where(ocu), w),
        "ing_p25_ocup":    w_quantile(ing.where(ocu), w, 0.25),
        "ing_p75_ocup":    w_quantile(ing.where(ocu), w, 0.75),
        "esc_media":       w_mean(num(d, "anios_esc"), w),
        "hrs_media_ocup":  w_mean(num(d, "hrsocup").where(ocu), w),
        "cobertura_ing_%": 100 * float(ing.where(ocu).notna().sum() / max(int(ocu.sum()), 1)),
    }
    if "emp_ppal" in d.columns:      # [1-2] formal/informal — confirma con ver_catalogo("emp_ppal")
        r["tasa_informalidad"] = w_share(num(d, "emp_ppal").eq(1), w, ocu)
    if "seg_soc" in d.columns:       # [1-3] condición de acceso a instituciones de salud
        r["acceso_salud"] = w_share(num(d, "seg_soc").eq(1), w, ocu)
    if "ingocup_sm" in d.columns:
        r["ing_mediana_sm"] = w_quantile(num(d, "ingocup_sm").where(ocu), w)
    return r

def agregar(df: pd.DataFrame, by: list, etiqueta: str) -> pd.DataFrame:
    by = [c for c in by if c in df.columns]
    if not by:
        print(f"  [!] {etiqueta}: faltan columnas de agrupación"); return pd.DataFrame()
    filas = []
    for llaves, d in df.groupby(by, dropna=False, observed=True):
        llaves = llaves if isinstance(llaves, tuple) else (llaves,)
        filas.append({**dict(zip(by, llaves)), **resumen_grupo(d)})
    out = pd.DataFrame(filas)
    out.columns = list(by) + [f"{etiqueta}_{c}" for c in out.columns[len(by):]]
    return out

AGG_ENT  = agregar(ana, ["cve_ent"], "ent")
AGG_MUN  = agregar(ana, ["cvegeo"],  "mun")     # cvegeo = entidad + municipio, llave INEGI
AGG_TLOC = agregar(ana, [T_LOC],     "tloc") if T_LOC else pd.DataFrame()

print(f"Entidad:   {len(AGG_ENT)} filas");  display(AGG_ENT.head())
print(f"Municipio: {len(AGG_MUN)} filas")
if len(AGG_MUN):
    display(AGG_MUN.sort_values("mun_n_muestra", ascending=False).head())
print(f"{T_LOC}: {len(AGG_TLOC)} filas");   display(AGG_TLOC)

Entidad:   32 filas


,cve_ent,ent_n_muestra,ent_pob_expandida,ent_pob_ocupada,ent_tasa_desocup,ent_tasa_particip,ent_ing_medio_ocup,ent_ing_mediana_ocup,ent_ing_p25_ocup,ent_ing_p75_ocup,ent_esc_media,ent_hrs_media_ocup,ent_cobertura_ing_%,ent_tasa_informalidad,ent_acceso_salud,ent_ing_mediana_sm
0,01,10757,1531396.0,698313.0,0.021569,0.592368,12088.064591,9890.0,7310.0,13125.827338,8.647653,41.966451,56.976744,0.445499,0.496958,0.448758
1,02,15450,3830329.0,1773880.0,0.023786,0.589794,14621.006023,12900.0,10750.0,17200.000000,8.730696,39.865591,57.182320,0.376519,0.554931,0.641312
2,03,7965,928591.0,484352.0,0.020823,0.669580,16309.879334,12900.0,8600.0,20000.000000,8.849209,39.602518,75.945428,0.389634,0.529596,1.121895
3,04,15600,935371.0,430913.0,0.027647,0.603625,10344.553875,8600.0,5160.0,12861.239437,8.096031,38.912133,79.449794,0.606095,0.331018,0.715580
4,05,15940,3590747.0,1606342.0,0.033857,0.595760,12407.420952,10750.0,8600.0,14000.000000,8.637595,40.657438,76.686014,0.343804,0.610148,0.897516


Municipio: 1038 filas


,cvegeo,mun_n_muestra,mun_pob_expandida,mun_pob_ocupada,mun_tasa_desocup,mun_tasa_particip,mun_ing_medio_ocup,mun_ing_mediana_ocup,mun_ing_p25_ocup,mun_ing_p75_ocup,mun_esc_media,mun_hrs_media_ocup,mun_cobertura_ing_%,mun_tasa_informalidad,mun_acceso_salud,mun_ing_mediana_sm
189,11020,8549,1786774.0,887225.0,0.031382,0.653375,11169.055952,9890.0,7000.0,12900.0,8.204735,39.294255,60.375587,0.465335,0.477915,0.575380
14,02004,6705,1913837.0,909281.0,0.022518,0.601254,14996.810723,13760.0,11180.0,17200.0,8.770862,39.262474,59.479787,0.361525,0.570674,0.672066
157,10005,6688,684980.0,316334.0,0.039794,0.614078,11175.798944,9460.0,7740.0,12900.0,9.071083,40.106928,74.513619,0.465021,0.479278,0.875078
102,07089,6626,335112.0,137411.0,0.038748,0.546740,8075.619019,6880.0,4000.0,10000.0,8.040012,41.533101,57.540364,0.640233,0.309757,0.269255
129,08037,6583,1611832.0,757490.0,0.021657,0.604296,14642.739276,12900.0,10750.0,16340.0,8.736012,36.912240,68.090615,0.276307,0.660294,0.801641


t_loc_tri: 4 filas


,t_loc_tri,tloc_n_muestra,tloc_pob_expandida,tloc_pob_ocupada,tloc_tasa_desocup,tloc_tasa_particip,tloc_ing_medio_ocup,tloc_ing_mediana_ocup,tloc_ing_p25_ocup,tloc_ing_p75_ocup,tloc_esc_media,tloc_hrs_media_ocup,tloc_cobertura_ing_%,tloc_tasa_informalidad,tloc_acceso_salud,tloc_ing_mediana_sm
0,1,228322,63665570.0,30721051.0,0.029999,0.601323,13119.793549,10750.0,7740.0,15050.0,9.571043,40.065478,65.618479,0.421750,0.508363,0.730536
1,2,53110,20071842.0,9292395.0,0.025223,0.600874,10328.233815,9000.0,6450.0,12900.0,8.335157,40.934283,68.116232,0.561385,0.380792,0.673137
2,3,55162,19425610.0,8467695.0,0.021658,0.573946,9209.094230,8200.0,5160.0,11180.0,7.358878,40.431199,68.681319,0.678532,0.267196,0.626174
3,4,70146,27987844.0,11412159.0,0.016132,0.550644,7752.957647,6880.0,3870.0,10000.0,6.098191,38.266352,69.288364,0.792502,0.165587,0.448758


In [28]:
# --- tamaño de muestra por municipio: crítico antes de usarlos como feature ---
if len(AGG_MUN):
    corte = pd.cut(AGG_MUN["mun_n_muestra"], [0, 30, 100, 300, 1000, 10**9],
                   labels=["<30", "30-99", "100-299", "300-999", "1000+"])
    dist = corte.value_counts().sort_index().rename("municipios").to_frame()
    dist["% del total"] = (100 * dist["municipios"] / len(AGG_MUN)).round(1)
    display(dist)
    print("La ENOE NO es representativa a nivel municipal. Los municipios con n<100 producen\n"
          "medianas de ingreso con varianza enorme: úsalos con encogimiento (shrinkage) hacia\n"
          "la media estatal, o quédate en entidad × tamaño de localidad × estrato.")

,municipios,% del total
mun_n_muestra,,
<30,25,2.4
30-99,444,42.8
100-299,340,32.8
300-999,159,15.3
1000+,70,6.7


La ENOE NO es representativa a nivel municipal. Los municipios con n<100 producen
medianas de ingreso con varianza enorme: úsalos con encogimiento (shrinkage) hacia
la media estatal, o quédate en entidad × tamaño de localidad × estrato.


In [29]:
# --- pegar el contexto geográfico de vuelta al nivel persona -----------------
uni_ctx = ana.copy()
pares = [(AGG_ENT, ["cve_ent"]), (AGG_MUN, ["cvegeo"])]
if len(AGG_TLOC):
    pares.append((AGG_TLOC, [T_LOC]))
for tabla, by in pares:
    if len(tabla):
        by = [c for c in by if c in uni_ctx.columns]
        uni_ctx = uni_ctx.merge(tabla, on=by, how="left", validate="m:1")

print(f"Dataset final: {len(uni_ctx):,} filas × {uni_ctx.shape[1]} columnas")

# Encogimiento empírico del ingreso municipal hacia la mediana estatal (James-Stein simple)
if {"mun_ing_mediana_ocup", "ent_ing_mediana_ocup", "mun_n_muestra"} <= set(uni_ctx.columns):
    n  = uni_ctx["mun_n_muestra"]
    k  = 100.0   # pseudo-conteo: n=100 pondera 50/50 municipio vs estado
    lam = n / (n + k)
    uni_ctx["mun_ing_mediana_shrunk"] = (
        lam * uni_ctx["mun_ing_mediana_ocup"] + (1 - lam) * uni_ctx["ent_ing_mediana_ocup"])
    print("Añadida 'mun_ing_mediana_shrunk' (encogida hacia la mediana estatal, k=100).")
    print("\n'cvegeo' es la llave de 5 dígitos del INEGI: úsala directo para pegar el Índice\n"
          "de Rezago Social del CONEVAL o proyecciones de CONAPO al mismo nivel municipal.")

Dataset final: 406,740 filas × 523 columnas
Añadida 'mun_ing_mediana_shrunk' (encogida hacia la mediana estatal, k=100).

'cvegeo' es la llave de 5 dígitos del INEGI: úsala directo para pegar el Índice
de Rezago Social del CONEVAL o proyecciones de CONAPO al mismo nivel municipal.


## 7. Diccionario del dataset unificado y exportación

In [30]:
mapa_pref = {v: k for k, v in PREFIJO.items()}

def origen(col: str):
    for p, m in mapa_pref.items():
        if col.startswith(p):
            return m, col[len(p):]
    if col in DF["sdem"].columns:
        return "sdem", col
    return "derivada", col

filas = []
d_idx = DICC.set_index(["modulo", "variable"])["descripcion"].to_dict()
for c in uni_ctx.columns:
    mod, orig = origen(c)
    filas.append({"columna": c, "modulo_origen": mod, "variable_original": orig,
                  "descripcion": d_idx.get((mod, orig), ""),
                  "dtype": str(uni_ctx[c].dtype),
                  "no_nulos_%": round(100 * uni_ctx[c].notna().mean(), 1),
                  "n_unicos": int(uni_ctx[c].nunique(dropna=True))})
DICC_UNI = pd.DataFrame(filas)
display(DICC_UNI.groupby("modulo_origen").size().rename("columnas").to_frame())
display(DICC_UNI.head(25))

,columnas
modulo_origen,
coe1,182
coe2,130
derivada,47
hog,30
sdem,115
viv,20


,columna,modulo_origen,variable_original,descripcion,dtype,no_nulos_%,n_unicos
0,r_def,sdem,r_def,Resultado definitivo de la entrevista,int64,100.0,1
1,cve_loc,sdem,cve_loc,Número de la localidad,string,0.0,0
2,cve_mun,sdem,cve_mun,Número de municipio según entidad,string,99.2,220
3,est,sdem,est,Estrato nacional y estatal,int64,100.0,4
4,est_d_tri,sdem,est_d_tri,Estrato de diseño trimestral,int64,100.0,723
5,est_d_men,sdem,est_d_men,Estrato de diseño mensual,object,100.0,724
6,cve_ageb,sdem,cve_ageb,Número de ageb del marco nacional,string,100.0,1
7,t_loc_tri,sdem,t_loc_tri,Tamaño de localidad trimestral,int64,100.0,4
8,t_loc_men,sdem,t_loc_men,Tamaño de localidad mensual,object,100.0,5
9,cd_a,sdem,cd_a,Ciudad autorrepresentada,string,100.0,45


In [31]:
stem = f"enoe_{ANIO}_{TRIM}t"
rutas = {}

try:
    p = OUT / f"{stem}_unificado.parquet"; uni_ctx.to_parquet(p, index=False); rutas["unificado (parquet)"] = p
except Exception as e:
    print("Parquet no disponible (instala pyarrow):", e)

p = OUT / f"{stem}_unificado.csv.gz"; uni_ctx.to_csv(p, index=False, compression="gzip"); rutas["unificado (csv.gz)"] = p
p = OUT / f"{stem}_agg_entidad.csv";  AGG_ENT.to_csv(p, index=False);  rutas["agregado entidad"] = p
p = OUT / f"{stem}_agg_municipio.csv"; AGG_MUN.to_csv(p, index=False); rutas["agregado municipio"] = p
p = OUT / f"{stem}_diccionario_unificado.csv"; DICC_UNI.to_csv(p, index=False); rutas["diccionario"] = p

for k, v in rutas.items():
    print(f"{k:26s} -> {v}  ({v.stat().st_size/1e6:.1f} MB)")

unificado (parquet)        -> salida_enoe/enoe_2026_1t_unificado.parquet  (71.4 MB)
unificado (csv.gz)         -> salida_enoe/enoe_2026_1t_unificado.csv.gz  (61.5 MB)
agregado entidad           -> salida_enoe/enoe_2026_1t_agg_entidad.csv  (0.0 MB)
agregado municipio         -> salida_enoe/enoe_2026_1t_agg_municipio.csv  (0.2 MB)
diccionario                -> salida_enoe/enoe_2026_1t_diccionario_unificado.csv  (0.0 MB)


## 8. Notas metodológicas para el modelo de estimación de ingresos

**Variables de ingreso disponibles y su jerarquía de confianza**

| Variable | Módulo | Notas |
|---|---|---|
| `ingocup` | SDEM | Ingreso mensual de la ocupación principal, ya derivado por INEGI. Rango 1-999998; **999998 es no especificado**. Es la variable de trabajo. |
| `ing_x_hrs` | SDEM | Ingreso por hora; separa precio del trabajo de intensidad. |
| `ing7c` | SDEM | Ingreso en rangos de salarios mínimos (1-7), incluye "no especificado". |
| `salario` / `zona` | SDEM | Salario mínimo mensual y zona salarial (frontera norte vs resto). El notebook deriva `ingocup_sm`. |
| `c2_p6b2` | **COE2** | **Monto declarado en pesos**, 6 dígitos. Rango 000001-999998, 999999. |
| `c2_p6b1` | **COE2** | **Periodicidad del pago** (1-8), *no* el monto. Trampa de nomenclatura. |
| `c2_p6c` | COE2 | Ingreso en múltiplos del salario mínimo (1-9), respuesta de rescate. |
| `c2_p7gcan` | COE2 | Monto del segundo trabajo. |
| `c2_p6e_c` / `c2_p6f_c` | COE2 | Municipio y estado **donde trabaja**, distinto de donde reside. |

`ingocup` ya incorpora la imputación por rangos de INEGI. Antes de modelar, revisa
`cobertura_ing_%` en los agregados: la no respuesta de ingreso **no es aleatoria** (se concentra
en los extremos de la distribución), lo cual es exactamente el mismo problema de sesgo de
selección que ya trabajas del lado crediticio. Los pesos `fac_tri` corrigen no respuesta de la
*unidad*, no del *ítem*.

`c2_p6e_c` / `c2_p6f_c` merecen atención aparte: permiten separar el municipio de residencia del
de trabajo. En zonas metropolitanas como la ZMG eso importa — el ingreso se genera donde se
trabaja, pero el domicilio del expediente de crédito es donde se reside.

**Cómo enlazar esto con datos de buró**

No hay llave persona-a-persona: la ENOE es anónima y por diseño muestral. Los usos viables son:

1. **Enriquecimiento contextual** — pegar los agregados por entidad / municipio / `t_loc` al
   expediente vía domicilio. Es lo mismo que ya haces con el Índice de Rezago Social del CONEVAL,
   pero con una variable dependiente directamente laboral en vez de un proxy de carencias.
2. **Estimación en dos muestras (two-sample / SMD)** — estimar en ENOE el mapeo
   `E[ingreso | edad, sexo, escolaridad, ocupación, geografía]` y aplicar esos coeficientes sobre
   los atributos del expediente. Requiere que las covariables estén medidas de forma comparable
   en ambas fuentes; es el cuello de botella real.
3. **Calibración y ranking** — usar los cuantiles ENOE por geografía como referencia externa para
   verificar si el modelo entrenado sobre solicitantes sobrestima el ingreso en zonas donde la
   población bancarizada no es representativa. Directamente relevante para el segmento no-hit.

**Advertencias de representatividad**

- La ENOE es representativa a nivel nacional, por entidad federativa, por las ciudades
  autorrepresentadas (`cd_a`) y por tamaño de localidad. **No a nivel municipio.** `cve_mun` y
  `cvegeo` existen en el microdato pero los agregados municipales no tienen validez inferencial:
  por eso el notebook reporta `n_muestra` y calcula una versión encogida hacia la media estatal.
- El diseño es de panel rotatorio de 5 visitas (`n_ent` 1-5). Un archivo trimestral mezcla las
  cinco cohortes. Para seguimiento longitudinal une trimestres con
  `cd_a + cve_ent + con + v_sel + n_hog + h_mud + n_ren` y valida con `eda` y `sex` que sea la
  misma persona — hay reemplazos de vivienda que rompen el enlace. `h_mud` marca hogares mudados.
- Para varianzas y errores estándar correctos usa el diseño complejo (`upm` como UPM,
  `est_d_tri` como estrato) con `samplics` o `statsmodels`, no la varianza simple.

**Marco legal.** ENOE es dato estadístico público bajo la LSNIEG, no dato personal. El punto de
cuidado no es el uso de la ENOE en sí, sino que un feature agregado por geografía puede
convertirse en proxy de características protegidas. Documenta el análisis de impacto disparado
antes de meter estos features a un score en producción.

In [32]:
# Vistazo final
print(uni_ctx.shape)
cols_interes = [c for c in ["cve_ent","cve_mun","cvegeo",T_LOC,"eda","sex","anios_esc","niv_ins",
                            "clase1","clase2","pos_ocu","emp_ppal","seg_soc","imssissste",
                            "ingocup","ingocup_sm","ing_x_hrs","hrsocup","c2_p6b2","c2_p6c",
                            "ent_ing_mediana_ocup","mun_ing_mediana_ocup","mun_ing_mediana_shrunk",
                            FAC] if c and c in uni_ctx.columns]
display(uni_ctx[cols_interes].head(15))
display(uni_ctx[cols_interes].describe(include="all").T)

(406740, 524)


,cve_ent,cve_mun,cvegeo,t_loc_tri,eda,sex,anios_esc,niv_ins,clase1,clase2,pos_ocu,emp_ppal,seg_soc,imssissste,ingocup,ingocup_sm,ing_x_hrs,hrsocup,c2_p6b2,c2_p6c,ent_ing_mediana_ocup,mun_ing_mediana_ocup,mun_ing_mediana_shrunk,fac_tri
0,09,007,09007,1,61.0,2,11.0,3,1,1,1,2,1,1,10000,1.043623,46.51163,50,10000.0,,10000.0,9030.000000,9076.701974,504
1,09,007,09007,1,80.0,2,12.0,3,2,3,0,0,0,0,0,0.000000,0.00000,0,NaN,,10000.0,9030.000000,9076.701974,1108
2,09,007,09007,1,72.0,1,10.0,3,2,4,0,0,0,0,0,0.000000,0.00000,0,NaN,,10000.0,9030.000000,9076.701974,506
3,09,007,09007,1,50.0,2,9.0,3,2,4,0,0,0,0,0,0.000000,0.00000,0,NaN,,10000.0,9030.000000,9076.701974,506
4,09,007,09007,1,50.0,2,6.0,2,1,1,1,2,1,1,0,0.000000,0.00000,36,NaN,1,10000.0,9030.000000,9076.701974,1186
5,09,010,09010,1,18.0,2,11.0,3,1,1,4,1,2,4,0,0.000000,0.00000,17,NaN,,10000.0,10000.000000,10000.000000,506
6,09,010,09010,1,64.0,1,9.0,3,2,4,0,0,0,0,0,0.000000,0.00000,0,NaN,,10000.0,10000.000000,10000.000000,506
7,09,010,09010,1,23.0,1,12.0,4,1,1,1,1,2,4,12900,1.346274,100.00000,30,12900.0,,10000.0,10000.000000,10000.000000,574
8,09,010,09010,1,55.0,2,15.0,4,1,1,1,2,1,2,10000,1.043623,58.13953,40,10000.0,,10000.0,10000.000000,10000.000000,574
9,09,011,09011,1,23.0,2,12.0,4,1,1,3,1,2,4,0,0.000000,0.00000,25,NaN,2,10000.0,10000.000000,10000.000000,1158


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
cve_ent,406740,32,07,20572,NaN,NaN,NaN,NaN,NaN,NaN,NaN
cve_mun,403296,220,002,19484,NaN,NaN,NaN,NaN,NaN,NaN,NaN
cvegeo,406740,1038,11020,8549,NaN,NaN,NaN,NaN,NaN,NaN,NaN
t_loc_tri,406740.0,NaN,NaN,NaN,1.919192,1.174851,1.0,1.0,1.0,3.0,4.0
eda,406692.0,NaN,NaN,NaN,35.410549,21.618284,0.0,17.0,33.0,52.0,98.0
sex,406740,2,2,211723,NaN,NaN,NaN,NaN,NaN,NaN,NaN
anios_esc,405693.0,NaN,NaN,NaN,8.539728,5.431549,0.0,6.0,9.0,12.0,24.0
niv_ins,406740.0,NaN,NaN,NaN,2.645398,1.262664,0.0,2.0,3.0,4.0,5.0
clase1,406740.0,NaN,NaN,NaN,1.214506,0.691464,0.0,1.0,1.0,2.0,2.0
clase2,406740.0,NaN,NaN,NaN,1.927706,1.562749,0.0,1.0,1.0,4.0,4.0
